# Get historical data of 11 industries

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))  # notebook lives in notebooks/

from backtest import ReturnCalculator
from strategy import MomentumSignalGenerator
from inverse_volatility_sizer import VolatilityWeightGenerator
from metrics import PerformanceEvaluator

start_date, end_date = '2025-01-01', '2025-12-31'

returns_generator = ReturnCalculator()
prices = returns_generator.load_data()  # full history, so signals are warmed up before 2025
prices

# Phase 2 - Signal generation

In [ ]:
signals = MomentumSignalGenerator(lookback_days=63).get_signals(prices)
signals.loc[start_date:end_date]

# Phase 3 - Position Sizing

In [ ]:
weights = VolatilityWeightGenerator(rolling_std_window=20).get_weights(prices)
weights.loc[start_date:end_date]

# Phase 4 - Backtest

Returns are close-to-close, matching the close prices the signals are built from.

In [ ]:
strategy_returns = returns_generator.get_returns(prices, signals, weights).loc[start_date:end_date]

metrics = PerformanceEvaluator()
equity_curve = metrics.equity_curve(strategy_returns)

print(f"Total Cumulative Return: {(equity_curve.iloc[-1] - 1) * 100:.2f}%")
print(f"Maximum Drawdown: {metrics.drawdown(strategy_returns) * 100:.2f}%")
print(f"Annualized Sharpe Ratio: {metrics.sharpe_ratio(strategy_returns):.2f}")

equity_curve.plot(title='Equity curve', figsize=(10, 4))
plt.show()